# Librerias 

In [1]:
import pandas as pd 
import numpy as np

# Datos 

In [2]:
datos_acciones = pd.read_csv(r"D:\\Datases_CD\\7mo\\Aprendizaje_Automatico\\Proyecto\\Datos_Fin.csv")
df = datos_acciones.sort_values(["Ticker", "Date"]).copy()

In [3]:
df.head()

,Date,Close,Ticker,Return
0,1995-12-05,0.296162,AAPL,0.000000
73,1995-12-06,0.290538,AAPL,-0.018989
146,1995-12-07,0.289132,AAPL,-0.004839
219,1995-12-08,0.295225,AAPL,0.021071
292,1995-12-11,0.289601,AAPL,-0.019049


# Ingeneria de caracteristicas

### Momentum: 

- `ret_5d` — Retorno acumulado 5 días

Mide cuánto ha cambiado el precio en la última semana aproximada:

$$
\text{ret\_5d}_t = \frac{P_t - P_{t-5}}{P_{t-5}}
$$

- `ret_20d` — Retorno acumulado 20 días

Mide el cambio de precio en el último mes bursátil:

$$
\text{ret\_20d}_t = \frac{P_t - P_{t-20}}{P_{t-20}}
$$

In [4]:
# Retorno acumulado últimos 5 días
df["ret_5d"]  = df.groupby("Ticker")["Close"].pct_change(5)
# Retorno acumulado últimos 20 días (~1 mes)
df["ret_20d"] = df.groupby("Ticker")["Close"].pct_change(20)

### Volatilidad: 

- `vol_5d` — Volatilidad 5 días

Es la desviación estándar de los retornos diarios de los últimos 5 días:

$$
\text{vol\_5d}_t = \sqrt{\frac{1}{5-1} \sum_{i=0}^{4} \left(r_{t-i} - \bar r_{5d}\right)^2}
$$

- `vol_20d` — Volatilidad 20 días

Desviación estándar de los retornos diarios de los últimos 20 días:

$$
\text{vol\_20d}_t = \sqrt{\frac{1}{20-1} \sum_{i=0}^{19} \left(r_{t-i} - \bar r_{20d}\right)^2}
$$

In [5]:
# Volatilidad a 5 días
df["vol_5d"] = (
    df.groupby("Ticker")["Return"]
      .rolling(5)
      .std()
      .reset_index(level=0, drop=True)
)

# Volatilidad a 20 días
df["vol_20d"] = (
    df.groupby("Ticker")["Return"]
      .rolling(20)
      .std()
      .reset_index(level=0, drop=True)
)

### Medias moviles y distancias:

- `dist_ma_20` — Distancia a la media móvil 20 días

Primero se calcula la media móvil de 20 días:

$$
\text{MA\_{20}}(t) = \frac{1}{20} \sum_{i=0}^{19} P_{t-i}
$$

Luego la **distancia relativa**:

$$
\text{dist\_ma\_20}_t = \frac{P_t - \text{MA\_{20}}(t)}{\text{MA\_{20}}(t)}
$$


Nos indica qué tan **“estirado”** está el precio respecto a su tendencia de 1 mes. 

- `dist_ma_60` — Distancia a la media móvil 60 días

Igual idea pero con una media móvil más larga (3 meses aprox.):

$$
\text{MA\_{60}}(t) = \frac{1}{60} \sum_{i=0}^{59} P_{t-i}
$$

$$
\text{dist\_ma\_60}_t = \frac{P_t - \text{MA\_{60}}(t)}{\text{MA\_{60}}(t)}
$$

Mide la posición del precio respecto a una **tendencia de mediano plazo**.  

In [6]:
# Media móvil 20 días
df["ma_20"] = (
    df.groupby("Ticker")["Close"]
      .rolling(20)
      .mean()
      .reset_index(level=0, drop=True)
)

# Media móvil 60 días
df["ma_60"] = (
    df.groupby("Ticker")["Close"]
      .rolling(60)
      .mean()
      .reset_index(level=0, drop=True)
)

# Distancias relativas
df["dist_ma_20"] = (df["Close"] - df["ma_20"]) / df["ma_20"]
df["dist_ma_60"] = (df["Close"] - df["ma_60"]) / df["ma_60"]

### `rsi_10` — Relative Strength Index (RSI 14 días): 

El RSI compara la fuerza de las subidas vs. las bajadas en los últimos 10 días.  
A grandes rasgos:

1. Se separan días de ganancia y de pérdida.
2. Se calcula el promedio de ganancias y de pérdidas en 14 días:
   $$
   RS_t = \frac{\text{avg\_gain}_{10}}{\text{avg\_loss}_{10}}
   $$
3. Luego:
   $$
   RSI_t = 100 - \frac{100}{1 + RS_t}
   $$

Nos mide la **fuerza interna del movimiento** reciente.  
- RSI alto (cerca de 70–80) → zona de posible sobrecompra.  
- RSI bajo (cerca de 20–30) → zona de posible sobreventa. 

In [7]:
window_rsi = 10
diff = df.groupby("Ticker")["Close"].diff()

gain = diff.clip(lower=0)
loss = -diff.clip(upper=0)

avg_gain = (
    gain.groupby(df["Ticker"])
        .rolling(window_rsi)
        .mean()
        .reset_index(level=0, drop=True)
)

avg_loss = (
    loss.groupby(df["Ticker"])
        .rolling(window_rsi)
        .mean()
        .reset_index(level=0, drop=True)
)

rs = avg_gain / avg_loss
df["rsi_10"] = 100 - (100 / (1 + rs))

df["rsi_10"] = df["rsi_10"].replace([np.inf, -np.inf], np.nan)
df["rsi_10"] = df["rsi_10"].clip(lower=0, upper=100)

### `stoch_k_10` — Stochastic Oscillator %K (14 días)

El %K compara el precio actual contra el rango mínimo–máximo de los últimos 10 días:

$$
\%K_t = 100 \times \frac{P_t - \min\left(P_{t-9}, \dots, P_t\right)}{\max\left(P_{t-9}, \dots, P_t\right) - \min\left(P_{t-9}, \dots, P_t\right)}
$$

Nos dice que si: “En el rango de precios de las últimas 2 semanas, ¿qué tan arriba o abajo estoy hoy?”.

- Valores cercanos a 100 → precio cerca del máximo reciente (posible sobrecompra).  
- Valores cercanos a 0 → precio cerca del mínimo reciente (posible sobreventa).  

De modo que esto ayuda al modelo a distinguir situaciones extremas dentro del rango local de precios.

In [8]:
window_stoch = 10

low10 = (
    df.groupby("Ticker")["Close"]
      .rolling(window_stoch)
      .min()
      .reset_index(level=0, drop=True)
)

high10 = (
    df.groupby("Ticker")["Close"]
      .rolling(window_stoch)
      .max()
      .reset_index(level=0, drop=True)
)

den = high10 - low10

df["stoch_k_10"] = 100 * (df["Close"] - low10) / den
df.loc[den == 0, "stoch_k_10"] = np.nan

### `excess_ret_20d_vs_spy` — Exceso de retorno vs. SPY 

Podemos comparar el activo contra el mercado:

$$
\text{excess\_ret\_{20d}}(t) = \text{ret\_{20d}}^{\text{activo}}(t) - \text{ret\_{20d}}^{\text{SPY}}(t)
$$

Mide la **“alpha” reciente** del activo frente al índice.  
Valores positivos → el activo ha superado al mercado en el último mes.  
Valores negativos → lo ha hecho peor que el mercado.  

Ayuda porque el modelo no solo ve si algo sube o baja, sino si aporta algo extra comparado con simplemente estar en el índice.

In [9]:
if "SPY" in df["Ticker"].unique():

    spy_20 = (
        df[df["Ticker"] == "SPY"][["Date", "ret_20d"]]
          .rename(columns={"ret_20d": "spy_ret_20d"})
    )

    df = df.merge(spy_20, on="Date", how="left")
    df["excess_ret_20d_vs_spy"] = df["ret_20d"] - df["spy_ret_20d"]

else:
    print(" No se encontró SPY en los tickers, no se crea 'excess_ret_20d_vs_spy'.")


In [10]:
df.head()

,Date,Close,Ticker,Return,ret_5d,ret_20d,vol_5d,vol_20d,ma_20,ma_60,dist_ma_20,dist_ma_60,rsi_10,stoch_k_10,spy_ret_20d,excess_ret_20d_vs_spy
0,1995-12-05,0.296162,AAPL,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1995-12-06,0.290538,AAPL,-0.018989,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1995-12-07,0.289132,AAPL,-0.004839,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1995-12-08,0.295225,AAPL,0.021071,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1995-12-11,0.289601,AAPL,-0.019049,NaN,NaN,0.016551,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Limpieza 

In [11]:
df["Date"] = pd.to_datetime(df["Date"])
ultima_fecha = df["Date"].max()
cutoff_date = ultima_fecha - pd.DateOffset(years=5)

df_5y = df[df["Date"] >= cutoff_date].copy()

In [12]:
df_5y_model = df_5y.dropna().copy()

Hacer el mapeo de asiganar una categoria 

In [13]:
category_map = {
    # Macro
    "^DJI": "Macro", "^GSPC": "Macro", "^NDX": "Macro", "^RUT": "Macro",
    "SPY": "Macro", "QQQ": "Macro", "IWM": "Macro",
    "EFA": "Macro", "EEM": "Macro",
    "IEF": "Macro", "SHY": "Macro", "TLT": "Macro",
    "CL=F": "Macro", "GC=F": "Macro", "NG=F": "Macro", "SI=F": "Macro",

    # Technology
    "AAPL": "Technology", "MSFT": "Technology", "GOOG": "Technology",
    "GOOGL": "Technology", "META": "Technology", "AMZN": "Technology",
    "CRM": "Technology", "ADBE": "Technology", "INTU": "Technology",
    "ORCL": "Technology", "NOW": "Technology", "PLTR": "Technology",
    "UBER": "Technology", "CSCO": "Technology", "ANET": "Technology",
    "ACN": "Technology",

    # Semiconductors
    "NVDA": "Semiconductors", "AMD": "Semiconductors", "INTC": "Semiconductors",
    "QCOM": "Semiconductors", "TXN": "Semiconductors", "AVGO": "Semiconductors",
    "MU": "Semiconductors", "KLAC": "Semiconductors", "LRCX": "Semiconductors",
    "AMAT": "Semiconductors", "ADI": "Semiconductors",

    # Financial
    "BAC": "Financial", "C": "Financial", "JPM": "Financial", "MS": "Financial",
    "GS": "Financial", "WFC": "Financial", "SCHW": "Financial",
    "BLK": "Financial", "BX": "Financial", "BRK-B": "Financial",
    "SPGI": "Financial", "AXP": "Financial", "COF": "Financial",
    "V": "Financial", "MA": "Financial", "PGR": "Financial", "CB": "Financial",

    # Healthcare
    "JNJ": "Healthcare", "LLY": "Healthcare", "MRK": "Healthcare",
    "PFE": "Healthcare", "ABT": "Healthcare", "AMGN": "Healthcare",
    "GILD": "Healthcare", "MDT": "Healthcare", "SYK": "Healthcare",
    "ISRG": "Healthcare", "TMO": "Healthcare", "MCK": "Healthcare",
    "VRTX": "Healthcare", "HCA": "Healthcare", "ABBV": "Healthcare",

    # Industrials
    "CAT": "Industrials", "DE": "Industrials", "HON": "Industrials",
    "ETN": "Industrials", "GE": "Industrials", "BA": "Industrials",
    "LMT": "Industrials", "RTX": "Industrials", "UNP": "Industrials",
    "GEV": "Industrials",

    # Energy
    "XOM": "Energy", "CVX": "Energy", "COP": "Energy", "NEE": "Energy",

    # Consumer
    "WMT": "Consumer", "COST": "Consumer", "HD": "Consumer",
    "LOW": "Consumer", "MCD": "Consumer", "PEP": "Consumer",
    "KO": "Consumer", "TJX": "Consumer", "DIS": "Consumer",

    # Telecom
    "T": "Telecom", "TMUS": "Telecom", "VZ": "Telecom",

    # REIT
    "PLD": "REIT", "WELL": "REIT",

    # Autos
    "TSLA": "Autos",

    # Cybersecurity
    "CRWD": "Cybersecurity", "PANW": "Cybersecurity",

    # Travel
    "BKNG": "Travel",
}


In [14]:
df_5y_model["categoria"] = df_5y_model["Ticker"].map(category_map)
df_5y_model["categoria"] = df_5y_model["categoria"].fillna("Otros")
df_5y_model.tail()

,Date,Close,Ticker,Return,ret_5d,ret_20d,vol_5d,vol_20d,ma_20,ma_60,dist_ma_20,dist_ma_60,rsi_10,stoch_k_10,spy_ret_20d,excess_ret_20d_vs_spy,categoria
762269,2025-11-18,2348.739990,^RUT,0.003143,-0.044560,-0.055855,0.013873,0.012095,2447.298486,2435.448665,-0.040272,-0.035603,34.744961,5.964424,-0.016699,-0.039156,Macro
762270,2025-11-19,2347.889893,^RUT,-0.000362,-0.041990,-0.042284,0.014209,0.011783,2442.115479,2435.270162,-0.038584,-0.035881,23.557890,5.568864,-0.007742,-0.034542,Macro
762271,2025-11-20,2305.110107,^RUT,-0.018221,-0.032678,-0.071516,0.011356,0.011769,2433.237988,2434.125330,-0.052657,-0.053003,23.901788,0.000000,-0.028626,-0.042890,Macro
762272,2025-11-21,2369.590088,^RUT,0.027973,-0.007805,-0.057244,0.019360,0.013301,2426.043994,2433.978333,-0.023270,-0.026454,38.217873,42.097025,-0.026903,-0.030341,Macro
762273,2025-11-24,2414.283447,^RUT,0.018861,0.031137,-0.042118,0.017910,0.014120,2420.736169,2434.776058,-0.002666,-0.008417,42.872611,71.275965,-0.024094,-0.018024,Macro


Nos quedamos solo con los ultimos dos años 

In [15]:
# Guardar en un csv 
df_5y_model.to_csv(r"D:\\Datases_CD\\7mo\\Aprendizaje_Automatico\\Proyecto\\datos_preprocesados_finales_5y.csv", index=False)